# Other Utilities — Reference

| Utility | Purpose |
|---|---|
| `.pt.cols()` | List column names sorted or in original order |
| `.pt.handle_missing()` | Fill string NaN with a marker, numeric NaN with 0, strip whitespace |
| `.pt.group_x()` | Add a group-level scalar back to every row (via `transform`) |
| `.pt.clean_columns()` | Clean column header names (strip, squeeze, fill, case, dedupe) |
| `.pt.replace_values()` | Replace values, optionally scoped to columns, exact or substring match |
| `df.to_clip()` | Copy DataFrame to clipboard (tab-separated, no index) |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']

In [2]:
# Alphabetical column list — useful for quick discovery
penguins.pt.cols()

['bill_depth_mm',
 'bill_length_mm',
 'body_mass_g',
 'flipper_length_mm',
 'island',
 'sex',
 'species']

In [3]:
# Descending alphabetical sort — useful when column names form a natural reverse order
penguins.pt.cols(ascending=False)

['species',
 'sex',
 'island',
 'flipper_length_mm',
 'body_mass_g',
 'bill_length_mm',
 'bill_depth_mm']

## `handle_missing()` — clean before aggregating
Fills string/category NaN with a visible marker (default `.`) and numeric NaN with `0`. Also strips leading/trailing whitespace from string columns.

In [4]:
# Default fillna='.': NaN sex becomes '.', NaN numerics become 0
# Then aggregate to see the '.' group alongside real groups
(penguins
 .pt.handle_missing()
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,.
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,.
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [5]:
# Custom fillna marker — makes missing group explicit in reports
( penguins
 .pt.handle_missing(fillna='Unknown')
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,Unknown
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,Unknown
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [6]:
# Use a single-char marker like '$' to make missing data stand out in reports
( penguins
 .pt.handle_missing(fillna='$')
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,$
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,$
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `group_x()` — broadcast a group aggregate back to every row
Unlike `agg_df` which collapses rows, `group_x` **adds a column** to the original DataFrame using `transform`. Useful for computing a group value alongside row-level detail (e.g., share of group total, deviation from group mean).

In [7]:
# Default: count per group (a='n') — adds column 'n' to every row
( penguins
 .pt.handle_missing()
 .pt.group_x(group=['species','island' ])
 .pt.select('species', 'island', 'sex', 'body_mass_g', 'n')
 .sample(10)
)


,species,island,sex,body_mass_g,n
252,Gentoo,Biscoe,Female,5050.0,124
120,Adelie,Torgersen,Female,3150.0,52
65,Adelie,Biscoe,Male,3950.0,44
236,Gentoo,Biscoe,Female,4150.0,124
223,Gentoo,Biscoe,Male,5700.0,124
137,Adelie,Dream,Male,3975.0,56
175,Chinstrap,Dream,Male,3800.0,68
308,Gentoo,Biscoe,Female,4875.0,124
288,Gentoo,Biscoe,Female,4700.0,124
188,Chinstrap,Dream,Female,3850.0,68


In [8]:
# instead of count, group_x with value + aggfunc: add group mean of body_mass_g to every row
(penguins
 .pt.group_x(group=['species'], v='body_mass_g', a='mean')
 .rename(columns={'x': 'species_mean_mass'})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,species_mean_mass
144,Adelie,Dream,37.3,16.8,192.0,3000.0,Female,3700.662252
176,Chinstrap,Dream,46.7,17.9,195.0,3300.0,Female,3733.088235
158,Chinstrap,Dream,46.1,18.2,178.0,3250.0,Female,3733.088235
193,Chinstrap,Dream,46.2,17.5,187.0,3650.0,Female,3733.088235
14,Adelie,Torgersen,34.6,21.1,198.0,4400.0,Male,3700.662252
312,Gentoo,Biscoe,45.5,14.5,212.0,4750.0,Female,5076.016260
27,Adelie,Biscoe,40.5,17.9,187.0,3200.0,Female,3700.662252
42,Adelie,Dream,36.0,18.5,186.0,3100.0,Female,3700.662252
40,Adelie,Dream,36.5,18.0,182.0,3150.0,Female,3700.662252
231,Gentoo,Biscoe,49.0,16.1,216.0,5550.0,Male,5076.016260


In [9]:
#with dropna=False
penguins.pt.group_x(group=['species', 'sex'],dropna=False)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,n
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,73
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,73
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,73
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,6
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,73
...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,5
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,58
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,61
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,58


In [10]:
#with default dropna=True
penguins.pt.group_x(group=['species', 'sex'])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,n
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,73.0
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,73.0
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,73.0
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,73.0
...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,58.0
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,61.0
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,58.0


## `df.to_clip()` — copy to clipboard
`df.to_clip()` copies the DataFrame to the system clipboard as tab-separated text (no index). Paste directly into Excel, Google Sheets, or any text editor. Invisible in notebook output — run it and then paste.

In [11]:
# df.to_clip() copies the current DataFrame to clipboard — paste into Excel/Sheets
# Run this cell then Cmd+V in any spreadsheet application
(penguins
 .pt.select('species', 'island', 'bill_length_mm', 'body_mass_g')
 .pt.agg_df(['species', 'island'], a=['mean', 'n'])
).to_clip()


## `clean_columns()` — clean messy header names
Cleans column header names in a fixed order: `strip` -> `strip_special` -> `squeeze` -> `fill` -> `case` -> `dedupe`. Handy after loading a spreadsheet export with inconsistent spacing/casing.

In [12]:
# messy headers: leading/trailing spaces, repeated internal spaces, mixed case
messy = penguins.rename(columns={
    'species': '  Species  ',
    'island': 'Island   Name',
    'body_mass_g': 'Body Mass (g)',
})

(messy
 .pt.clean_columns(strip=True, squeeze=True, strip_special=True, fill='_', case='lower')
 .pt.cols(ascending=None)
)

['species',
 'island_name',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex']

In [13]:
# Method chain: clean names, then display them in original column order
(
    penguins
    .pt.clean_columns(case='proper')
    .pt.cols(ascending=None)
)


['Species',
 'Island',
 'Bill_Length_Mm',
 'Bill_Depth_Mm',
 'Flipper_Length_Mm',
 'Body_Mass_G',
 'Sex']

## `replace_values()` — replace values, whole-cell or substring
Wraps pandas `replace()`. `exact=True` (default) only replaces a cell whose entire value matches a key exactly; `exact=False` replaces a key anywhere it occurs as a substring, leaving the rest of the cell untouched. Optionally scope to specific columns with `c=`. Parameter names (`v=`, `c=`, `exact=`) match the `-replace_values` CLI flag.

In [14]:
# exact=True (default): only cells that equal 'Adelie'/'Male' exactly are replaced
(penguins
 .pt.replace_values({'Adelie': 'Adelie (renamed)', 'Male': 'M'})
 .pt.select('species', 'sex')
 .drop_duplicates()
)

,species,sex
0,Adelie (renamed),M
1,Adelie (renamed),Female
3,Adelie (renamed),NaN
152,Chinstrap,Female
153,Chinstrap,M
220,Gentoo,Female
221,Gentoo,M
246,Gentoo,NaN


In [15]:
# c= restricts the replace to specific columns — 'island' untouched even though
# it never contained 'Male' anyway, this shows the scoping mechanism itself
(penguins
 .pt.replace_values({'Male': 'M', 'Female': 'F'}, c='sex')
 .pt.select('island', 'sex')
 .drop_duplicates()
)

,island,sex
0,Torgersen,M
1,Torgersen,F
3,Torgersen,NaN
20,Biscoe,F
21,Biscoe,M
30,Dream,F
31,Dream,M
47,Dream,NaN
246,Biscoe,NaN


In [16]:
# exact=False: substring match anywhere in the cell, rest of the cell untouched
import pandas as pd

notes = pd.DataFrame({'note': ['a magician performed', 'not a magician exactly', 'no match here']})
notes.pt.replace_values({'a magician': 'the magic'}, exact=False)

,note
0,the magic performed
1,not the magic exactly
2,no match here
